Install Dependencies

In [ ]:
!pip install transformers flask flask-cors pyngrok peft

Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Log in to Hugging Face

In [ ]:
from huggingface_hub import login
login()


Load Model and Tokenizer

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

# Path to your LoRA adapter in Google Drive
peft_path = "/content/drive/MyDrive/mistral-whatsapp-final"

# Load base Mistral model from Hugging Face (requires login)
base_model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-v0.1",
    device_map="auto",
    torch_dtype=torch.float16
)

# Load tokenizer from the base model
tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-v0.1")

# Load the fine-tuned LoRA adapter
model = PeftModel.from_pretrained(base_model, peft_path)

# Move to GPU or CPU
model.to("cuda" if torch.cuda.is_available() else "cpu")


Set up Flask Server

In [ ]:
from flask import Flask, request, jsonify
from flask_cors import CORS

app = Flask(__name__)
CORS(app)

@app.route("/generate", methods=["POST"])
def generate():
    data = request.json
    prompt = data.get("prompt", "")

    # Add Gurt's name to steer the model into speaking as the bot
    prompt += "\nGurt:"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=True,
            temperature=0.85,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract just Gurt's reply (first line only)
    if "Gurt:" in full_output:
        reply = full_output.split("Gurt:")[-1].strip().split("\n")[0]
    else:
        reply = full_output[len(prompt):].strip().split("\n")[0]

    return jsonify({"response": reply})


Authenticate Ngrok Authtoken

In [ ]:
!ngrok config add-authtoken 2wvJt5WodUEHn0II3UzJL09WmwD_6cCoyQvPMqueu4uyuJyQq

Run with Ngrok

In [ ]:
from pyngrok import ngrok

public_url = ngrok.connect(5000)
print("✅ Public URL:", public_url)

app.run(port=5000)
